# Boston Marathons Performance X Weather Conditions

<img src="images/marathon_image.png" alt="Marathon image" title="Titre de l'image" width="300" />


## Welcome to our data project !

Our project investigates the relationship between Boston Marathon runner performance and weather conditions between 2000 and 2019.  
Specifically, we address three analytical questions :
1. (request 1)
2. (request 2)
3. (request 3)

To conduct this analysis, we built an integrated analytical database using five automated Airflow pipelines (three for data ingestion, one for data transformation and one for production data).
Our final database combines data from three sources :
- A GitHub repository containing the CSV files of the results of each Boston Marathon edition ;
- Wikidata to collect the dates of each Boston marathon edition ;
- Meteostat (a python library) to collect daily weather data for Boston from 2000 to 2019.

Want to see the result of our analysis ?
1. Execute each code cell in order.
2. Make sure to wait for each cell to finish running before launching the next one.
3. Enjoy ! 🏃‍♂️😉

## Authentication

In [1]:
import requests

headers = {
    "Accept": "application/json",
    "Content-Type": "application/json"
}
body = {
  "username": "airflow",
  "password": "airflow"
}
url = f"http://airflow-apiserver:8080/auth/token"
r = requests.post(url, headers=headers, json=body)

jwt_token = r.json().get("access_token")

## Create connections between Airflow and databases

In [2]:
import requests

# MongoDB
headers = {
    "Accept": "application/json",
    "Authorization": f"Bearer {jwt_token}",
    "Content-Type": "application/json"
}
body = {
    "connection_id": "mongo_default",
    "conn_type": "mongo",
    "description": "mongo_default",
    "host": "mongo",
    "login": "admin",
    "schema": "project",
    "port": 27017,
    "password": "admin",
    "extra":  "{\"srv\": false, \"ssl\": false, \"allow_insecure\": false, \"authSource\": \"admin\"}"
}
url = f"http://airflow-apiserver:8080/api/v2/connections"
r = requests.post(url, headers=headers, json=body)

# Postgres default
body = {
    "connection_id": "postgres_default",
    "conn_type": "postgres",
    "description": "postgres_default",
    "host": "postgres",
    "login": "airflow",
    "schema": "airflow",
    "port": 5432,
    "password": "airflow",
}
url = f"http://airflow-apiserver:8080/api/v2/connections"
r = requests.post(url, headers=headers, json=body)

# Postgres Staging Zone
body = {
    "connection_id": "postgres_staging",
    "conn_type": "postgres",
    "description": "postgres_staging",
    "host": "postgres",
    "login": "airflow",
    "schema": "staging",
    "port": 5432,
    "password": "airflow",
}
url = f"http://airflow-apiserver:8080/api/v2/connections"
r = requests.post(url, headers=headers, json=body)

# Postgres Production Zone
body = {
    "connection_id": "postgres_production",
    "conn_type": "postgres",
    "description": "postgres_production",
    "host": "postgres",
    "login": "airflow",
    "schema": "production",
    "port": 5432,
    "password": "airflow",
}
url = f"http://airflow-apiserver:8080/api/v2/connections"
r = requests.post(url, headers=headers, json=body)

## Ingestion Pipelines

In [3]:
import requests
from datetime import datetime, timezone
import pytz
import time

DAG_INGESTION_IDS = ["ingest_marathons", "ingest_marathons_date", "ingest_weather"]

headers = {
    "Accept": "application/json",
    "Authorization": f"Bearer {jwt_token}",
    "Content-Type": "application/json"
}

dag_runs = {}
for dag in DAG_INGESTION_IDS :
    dt = datetime.now(timezone.utc)
    logical_date = dt.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"
    body = {
        "logical_date": logical_date,
        "conf": {}
    }
    url = f"http://airflow-apiserver:8080/api/v2/dags/{dag}/dagRuns"
    r = requests.post(url, headers=headers, json=body)
    resp_json = r.json()
    dag_runs[dag] = {
        "dag_run_id": resp_json["dag_run_id"],
        "dag_id" : resp_json["dag_id"],
        "state": ""
    }

all_done = False
while not all_done:
    print("The DAGs are running...")
    time.sleep(10)
    all_done = True
    for dag, info in dag_runs.items():
        if info["state"] not in ["success", "failed"]:
            url = f"http://airflow-apiserver:8080/api/v2/dags/{info['dag_id']}/dagRuns/{info['dag_run_id']}"
            r = requests.get(url, headers=headers)
            resp_json = r.json()
            state = resp_json["state"]
            info["state"] = state
            if state not in ["success", "failed"]:
                all_done = False    

for dag, info in dag_runs.items():
    if info["state"] == "success":
        print(f"DAG {dag} has been run successfully !")
    else:
        print(f"Error during the execution of the DAG {dag} !")

The DAGs are running...
The DAGs are running...
The DAGs are running...
The DAGs are running...
The DAGs are running...
The DAGs are running...
The DAGs are running...
The DAGs are running...
The DAGs are running...
The DAGs are running...
The DAGs are running...
The DAGs are running...
The DAGs are running...
The DAGs are running...
The DAGs are running...
DAG ingest_marathons has been run successfully !
DAG ingest_marathons_date has been run successfully !
DAG ingest_weather has been run successfully !


## Staging pipeline

In [5]:
import requests
from datetime import datetime, timezone
import pytz
import time

headers = {
    "Accept": "application/json",
    "Authorization": f"Bearer {jwt_token}",
    "Content-Type": "application/json"
}

dag = "staging_data"
dt = datetime.now(timezone.utc)
logical_date = dt.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"
body = {
    "logical_date": logical_date,
    "conf": {}
}
url = f"http://airflow-apiserver:8080/api/v2/dags/{dag}/dagRuns"
r = requests.post(url, headers=headers, json=body)

resp_json = r.json()
dag_run_id = resp_json["dag_run_id"]
dag_id = resp_json["dag_id"]
state = ""
while state != "success" and state != "failed":
    print("The DAG is running...")
    time.sleep(10)
    url = f"http://airflow-apiserver:8080/api/v2/dags/{dag_id}/dagRuns/{dag_run_id}"
    r = requests.get(url, headers=headers)
    resp_json = r.json()
    state = resp_json["state"] 

if state == "success":
    print("The DAG has been run successfully !")
else :
    print("Error during the execution of the DAG !")

The DAG is running...
The DAG is running...
The DAG is running...
The DAG is running...
The DAG is running...
The DAG is running...
The DAG is running...
The DAG is running...
The DAG is running...
The DAG is running...
The DAG is running...
The DAG is running...
The DAG is running...
The DAG is running...
The DAG is running...
The DAG has been run successfully !


## Production pipeline

In [7]:
import requests
from datetime import datetime, timezone
import pytz
import time

headers = {
    "Accept": "application/json",
    "Authorization": f"Bearer {jwt_token}",
    "Content-Type": "application/json"
}

dag = "production_dag"
dt = datetime.now(timezone.utc)
logical_date = dt.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"
body = {
    "logical_date": logical_date,
    "conf": {}
}
url = f"http://airflow-apiserver:8080/api/v2/dags/{dag}/dagRuns"
r = requests.post(url, headers=headers, json=body)

resp_json = r.json()
dag_run_id = resp_json["dag_run_id"]
dag_id = resp_json["dag_id"]
state = ""
while state != "success" and state != "failed":
    print("The DAG is running...")
    time.sleep(10)
    url = f"http://airflow-apiserver:8080/api/v2/dags/{dag_id}/dagRuns/{dag_run_id}"
    r = requests.get(url, headers=headers)
    resp_json = r.json()
    state = resp_json["state"]
    
if state == "success":
    print("The DAG has been run successfully !")
else :
    print("Error during the execution of the DAG !")

The DAG is running...
The DAG is running...
The DAG is running...
The DAG is running...
The DAG is running...
The DAG is running...
The DAG is running...
The DAG is running...
Error during the execution of the DAG !


## Execution of analytical request

In [12]:
import psycopg2

conn = psycopg2.connect(
    host="postgres",
    port=5432,
    database="production",
    user="airflow",
    password="airflow"
)

cur = conn.cursor()
cur.execute("""
    SELECT 
        d.year,
        l.city,
        w.pressure,
        r.gender,
        AVG(f.time)::time(0) as average_finish_time
    FROM factraceresult f
    JOIN dimweather w ON f.weatherkey = w.weatherkey
    JOIN dimrunner r ON f.runnerkey = r.runnerkey
    JOIN dimdate d ON f.datekey = d.datekey
    JOIN dimlocation l ON f.locationkey = l.locationkey
    WHERE f.overallranking <= 100
    GROUP BY 
        d.year, 
        l.city, 
        w.pressure, 
        r.gender
    ORDER BY 
        w.pressure DESC;
""")

rows = cur.fetchall()

# Afficher un en-tête pour la lisibilité
print(f"{'Year':<6} | {'City':<10} | {'Pressure':<10} | {'Gender':<6} | {'Avg Time'}")
print("-" * 60)

# Boucle pour afficher chaque résultat
for row in rows:
    year, city, pressure, gender, time = row
    print(f"{year:<6} | {city:<10} | {pressure:<10} | {gender:<6} | {time}")

cur.close()
conn.close()

Year   | City       | Pressure   | Gender | Avg Time
------------------------------------------------------------
2013   | Boston     | 1032.4     | F      | 02:38:23
2013   | Boston     | 1032.4     | M      | 02:26:37
2008   | Boston     | 1027.8     | F      | 02:29:08
2008   | Boston     | 1027.8     | M      | 02:27:23
2009   | Boston     | 1024.5     | F      | 02:32:58
2009   | Boston     | 1024.5     | M      | 02:27:07
2000   | Boston     | 1023.7     | F      | 02:31:29
2000   | Boston     | 1023.7     | M      | 02:27:52
2016   | Boston     | 1022.5     | F      | 02:33:29
2016   | Boston     | 1022.5     | M      | 02:30:49
2014   | Boston     | 1019.2     | F      | 02:22:17
2014   | Boston     | 1019.2     | M      | 02:23:22
2015   | Boston     | 1018.0     | F      | 02:26:43
2015   | Boston     | 1018.0     | M      | 02:24:43
2005   | Boston     | 1017.2     | F      | 02:32:34
2005   | Boston     | 1017.2     | M      | 02:30:49
2003   | Boston     | 1017.0     | F  